# Item-Based Collaborative Filtering: The Matrix, Flipped

User-based CF found the users whose taste matches yours and borrowed their ratings. Item-based CF flips the matrix: instead of matching users to users, it matches movies to movies. A movie is scored for a user by the ratings that user gave to the movie's nearest sister movies. Same family, same Pearson math, a different axis, and a different set of costs. This stage builds the item-item model, sweeps the knob that actually controls it, and checks the honest state of the lab after four stages.

## The item-based family

The two members of the nearest-neighbor family are transposes of each other. User-based puts users on the rows and computes user-user similarity over the items both rated. Item-based puts movies on the rows and computes item-item similarity over the users who rated both. The score for a user and a movie is the movie's mean rating plus the deviation of the user's own ratings on similar movies, weighted by similarity. No movie feature and no user feature ever enter the computation.

Item-based carries the two documented weaknesses in a different shape. First, the item-item matrix is quadratic in the number of movies: with 3,667 movies that is 13.4 million pair comparisons, which is why the model keeps only the k nearest neighbors per movie. Second, the skew problem persists: a correlation between two movies computed over a handful of shared users is almost meaningless, and the minimum co-rated threshold matters even more here because movies accumulate raters much faster than users accumulate movies.

### Reference
The collaborative filtering theory in this notebook follows: Schafer, J.B., Frankowski, D., Herlocker, J., Sen, S. (2007). Collaborative Filtering Recommender Systems. In: The Adaptive Web. Lecture Notes in Computer Science, vol 4321. Springer, Berlin, Heidelberg, pp. 291-324. https://doi.org/10.1007/978-3-540-72079-9_9

## Setup

Same loader, same time-based split, same evaluation as the previous three stages, so every number is comparable.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "recommendation_lab").is_dir():
    ROOT = ROOT.parent
    if ROOT == ROOT.parent:
        raise RuntimeError("could not locate recommendation_lab package")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from recommendation_lab.data.loader import load_ml_1m
from recommendation_lab.data.split import time_base_split
from recommendation_lab.evaluation.evaluate import evaluate_predictions, evaluate_ranking
from recommendation_lab.recommenders.content_based import ContentBasedRecommender
from recommendation_lab.recommenders.popularity import PopularityRecommender
from recommendation_lab.recommenders.user_based_cf import UserBasedRecommender
from recommendation_lab.recommenders.item_based_cf import ItemBasedRecommender

data = load_ml_1m()
ratings, movies = data["ratings"], data["movies"]

split = time_base_split(ratings)
train, test = split.train, split.test
pd.set_option("display.max_colwidth", 80)
print(f"train: {len(train):,} ratings | test: {len(test):,} ratings")

Dataset 'ml-1m' already present at /Users/kayceejenz/Documents/experiment/recommendation-system-lab/data/ml-1m


train: 797,758 ratings | test: 202,397 ratings


## The signal: who rated what

The item-based model reads the same user-item matrix, transposed. Each row is now a movie, each column a user, and a rating sits where that user rated that movie. Two movies are similar when the columns they both fill agree.

In [2]:
from scipy import sparse

user_ids = train["user_id"].unique()
item_ids = train["movie_id"].unique()

u_idx = pd.Series(np.arange(len(user_ids)), index=user_ids)
i_idx = pd.Series(np.arange(len(item_ids)), index=item_ids)

rows = u_idx[train["user_id"]].to_numpy()
cols = i_idx[train["movie_id"]].to_numpy()
vals = train["rating"].to_numpy()
R = sparse.csr_matrix((vals, (rows, cols)), shape=(len(user_ids), len(item_ids)))
Rt = R.T.tocsr()

print(f"user-item matrix: {R.shape[0]:,} users x {R.shape[1]:,} movies")
print(f"item-item matrix: {Rt.shape[0]:,} movies x {Rt.shape[1]:,} users (the transpose)")
print(f"ratings per movie: mean {np.asarray(Rt.getnnz(axis=1)).mean():.1f} | median {np.median(np.asarray(Rt.getnnz(axis=1))):.0f}")

user-item matrix: 6,040 users x 3,667 movies
item-item matrix: 3,667 movies x 6,040 users (the transpose)
ratings per movie: mean 217.6 | median 91


**Findings:**
The rows are the same 3,667 movies, but the row profile is denser than the user profile: 218 raters per movie on average versus 132 ratings per user. Every movie is a column of the user-user matrix from the last stage, and the columns are dense enough to lean on. The skew problem does not disappear, but it changes shape, as the next section shows.

## Pearson similarity over co-rated users

Two movies are similar when the same users rate them alike. The Pearson correlation over co-rated users does exactly this: subtract each movie's mean rating, multiply the centered ratings user by user, and normalize by the two standard deviations.

    sim(a, b) = sum_u (r_ua - mean_a)(r_ub - mean_b) / (std_a * std_b)

Restrict the sum to the users who rated both movies, and the matrix product accumulates co-rated contributions in one pass. The recommender builds this item-item matrix at fit time. Star Wars' nearest neighbors show what the matrix finds.

In [3]:
ibcf = ItemBasedRecommender(k_neighbors=5, min_co_ratings=100).fit(train)

i = int(np.where(ibcf.item_ids == 260)[0][0])
row = ibcf._sim[i].toarray().ravel()
nbrs = np.argsort(row)[::-1][:5]

print(f"Star Wars: Episode IV - A New Hope (1977), mean rating {ibcf.item_mean[i]:.2f}")
print("nearest neighbors (movie id, Pearson sim, co-rated):")
for j in nbrs:
    if row[j] != 0:
        common = (Rt[i] != 0).multiply(Rt[j] != 0).nnz
        print(f"  movie {ibcf.item_ids[j]:6d}  sim {row[j]:+.3f}  co-rated {common:4d}")

Star Wars: Episode IV - A New Hope (1977), mean rating 4.46
nearest neighbors (movie id, Pearson sim, co-rated):
  movie   1196  sim +0.655  co-rated 2039
  movie   1210  sim +0.572  co-rated 1812
  movie   1198  sim +0.433  co-rated 1682
  movie   2628  sim +0.369  co-rated 1336
  movie   3102  sim +0.346  co-rated  120


**Findings:**
The item-item matrix finds franchises: Star Wars' closest sisters are its own sequels and Raiders of the Lost Ark, and every one of them rests on a thousand or more shared raters. Compare this with the user side from the last stage, where a correlation of 0.98 sat on five shared movies. Item correlations are lower in magnitude (0.66 at the top) but they stand on far more evidence. The sparsity that made user similarity guesswork is milder here.

## The skew problem in numbers

How widespread is the problem on the item side? Across all pairs of movies, look at how many users they co-rated and what magnitude of correlation that produces. If the correlation were meaningful, it would stay moderate regardless of sample size. The buckets now run higher, because movies accumulate raters quickly.

In [4]:
binary = (Rt > 0).astype(np.int8)
common = (binary @ binary.T).toarray()
np.fill_diagonal(common, 0)

# Pearson over co-rated users, replicated from the recommender
centered = Rt.copy().astype(np.float32)
item_of_row = np.repeat(np.arange(len(item_ids)), np.diff(Rt.indptr))
centered.data -= (np.asarray(Rt.sum(axis=1)).ravel() / np.asarray(Rt.getnnz(axis=1)).ravel())[item_of_row]
bin32 = (Rt > 0).astype(np.float32)
numer = (centered @ centered.T).toarray()
denom = np.sqrt(((centered.multiply(centered)) @ bin32.T).toarray() * (bin32 @ (centered.multiply(centered)).T).toarray())
np.maximum(denom, 1e-9, out=denom)
sim = np.divide(numer, denom, out=np.zeros_like(numer), where=denom > 1e-9)
np.fill_diagonal(sim, 0)

ri, ci = np.triu_indices(len(item_ids), 1)
co = common[ri, ci]
sc = np.abs(sim[ri, ci])

for lo in [5, 10, 20, 50, 100]:
    m = co == lo
    print(f"co-rated = {lo:4d}: {m.sum():>9,} pairs | mean |corr| {sc[m].mean():.3f} | |corr| > 0.8 in {(sc[m] > 0.8).mean():6.2%}")
m = co >= 100
print(f"co-rated >= 100: {m.sum():>9,} pairs | mean |corr| {sc[m].mean():.3f} | |corr| > 0.8 in {(sc[m] > 0.8).mean():6.2%}")

co-rated =    5:   234,144 pairs | mean |corr| 0.409 | |corr| > 0.8 in  8.14%
co-rated =   10:   114,911 pairs | mean |corr| 0.305 | |corr| > 0.8 in  1.10%
co-rated =   20:    50,611 pairs | mean |corr| 0.241 | |corr| > 0.8 in  0.09%
co-rated =   50:    13,982 pairs | mean |corr| 0.192 | |corr| > 0.8 in  0.00%
co-rated =  100:     4,134 pairs | mean |corr| 0.170 | |corr| > 0.8 in  0.00%
co-rated >= 100:    90,277 pairs | mean |corr| 0.168 | |corr| > 0.8 in  0.00%


**Findings:**
The skew is the same disease with the same dose-response. Pairs of movies who shared five raters average |0.41| correlation and 8% sit above 0.8; pairs who shared a hundred raters average |0.17| and none reach 0.8. The magic number shifts because movies fill their columns fast, but the mechanism is unchanged: high correlations on little shared evidence are noise, and a model that trusts them is trusting coin flips.

## How many neighbors?

For user-based, the number of neighbors was the decisive knob. For item-based the item-item matrix is denser and better supported, so the neighborhood size should matter less. This is the same sweep, same split, same evaluation.

In [5]:
cols = ["precision@10", "recall@10", "map@10", "ndcg@10", "hit_rate@10"]
rows_k = []
for k in [5, 10, 20, 50, 100]:
    m = ItemBasedRecommender(k_neighbors=k, min_co_ratings=100).fit(train)
    r = evaluate_ranking(m, train, test, k=10)
    rmse = evaluate_predictions(m, test)["rmse"]
    rows_k.append({"k": k, **{c: r[c] for c in cols}, "rmse": rmse})
print(pd.DataFrame(rows_k).round(4).to_string(index=False))

  k  precision@10  recall@10  map@10  ndcg@10  hit_rate@10   rmse
  5        0.0427     0.0159  0.0179   0.0219       0.2813 1.1139
 10        0.0399     0.0144  0.0164   0.0202       0.2687 1.0866
 20        0.0396     0.0146  0.0164   0.0199       0.2629 1.0540
 50        0.0362     0.0136  0.0145   0.0185       0.2449 1.0160
100        0.0354     0.0140  0.0139   0.0185       0.2397 0.9988


**Findings:**
Neighborhood size barely moves the needle: every ranking metric drifts down only mildly as k grows from 5 to 100, and RMSE actually improves. This is the opposite of the user side, where each extra neighbor added noise and ranking fell steadily. Item similarities are supported by hundreds of raters, so the model can afford a crowd. The default k=5 is kept as a middle ground, nearly the best of the sweep and cheap to serve.

## How much support?

If k barely matters, the other threshold should matter a lot. The minimum co-rated count controls how much shared evidence a similarity must rest on before the model trusts it. For items, five shared users is nearly worthless, so the sweep runs higher. The default in this stage is min_co_ratings=100.

In [6]:
rows_minco = []
for minco in [5, 20, 50, 100, 300, 500]:
    m = ItemBasedRecommender(k_neighbors=5, min_co_ratings=minco).fit(train)
    r = evaluate_ranking(m, train, test, k=10)
    rmse = evaluate_predictions(m, test)["rmse"]
    rows_minco.append({"min_co_ratings": minco, **{c: r[c] for c in cols}, "rmse": rmse})
print(pd.DataFrame(rows_minco).round(4).to_string(index=False))

 min_co_ratings  precision@10  recall@10  map@10  ndcg@10  hit_rate@10   rmse
              5        0.0063     0.0014  0.0022   0.0021       0.0520 1.1449
             20        0.0173     0.0055  0.0063   0.0077       0.1346 1.1275
             50        0.0304     0.0113  0.0116   0.0151       0.2228 1.1162
            100        0.0427     0.0159  0.0179   0.0219       0.2813 1.1139
            300        0.0613     0.0234  0.0275   0.0325       0.3616 1.1316
            500        0.0745     0.0279  0.0348   0.0385       0.3995 1.1415


**Findings:**
This is the knob that controls the model. At the user-based default of five shared users, the item model is nearly useless: hit rate 0.05, precision 0.006. Raising the floor to 100 co-rated users lifts hit rate to 0.28, and 500 takes it to 0.40. The reason is the support histogram from the skew section: with 218 raters per movie on average, a floor of five admits almost every pair, including millions of noise correlations. The next section checks what the higher floors are actually trading.

## The popularity pull

Raising the support floor keeps only pairs of well-popular movies, which risks collapsing the model toward the popularity baseline. To see that drift, measure how often each configuration's top-10 overlaps the popularity top-100.

In [7]:
popularity = PopularityRecommender().fit(train)
top100 = set(popularity.recommend(1, k=100))
sample_users = np.random.default_rng(0).choice(train["user_id"].unique(), 200, replace=False)

for minco in [5, 100, 300, 500, 800]:
    m = ItemBasedRecommender(k_neighbors=5, min_co_ratings=minco).fit(train)
    overlap = np.mean([
        len(set(m.recommend(int(u), k=10)) & top100) / 10 for u in sample_users
    ])
    print(f"min_co_ratings={minco:4d}: {100 * overlap:5.1f}% of item-based top-10 also in popularity top-100")

min_co_ratings=   5:   0.5% of item-based top-10 also in popularity top-100


min_co_ratings= 100:  10.6% of item-based top-10 also in popularity top-100


min_co_ratings= 300:  23.1% of item-based top-10 also in popularity top-100


min_co_ratings= 500:  38.3% of item-based top-10 also in popularity top-100


min_co_ratings= 800:  69.1% of item-based top-10 also in popularity top-100


**Findings:**
The drift is real. At the low floor the item model is disjoint from popularity; at a floor of 800 nearly seven of every ten recommendations are popularity's own picks. Part of the min_co_ratings gain is genuine signal, but part is the model quietly turning into the popularity recommender it was built to beat. The default of 100 sits where the model has real ranking power (hit rate 0.28) without yet blending into the baseline (11% overlap).

## Population-wide agreement

The second documented weakness of user-based was population-wide agreement: everyone loves the blockbuster, so liking it says nothing about the user. The literature's fix is to weight ratings so that agreement on common things counts less. The item-based transpose weights the users: a prolific user who rates everything contributes agreement about almost any pair, so they get a quieter voice. The recommender supports this as `weighting="popularity"`.

In [8]:
rows_w = []
for w in ["none", "popularity"]:
    m = ItemBasedRecommender(k_neighbors=5, min_co_ratings=100, weighting=w).fit(train)
    r = evaluate_ranking(m, train, test, k=10)
    rmse = evaluate_predictions(m, test)["rmse"]
    rows_w.append({"weighting": w, **{c: r[c] for c in cols}, "rmse": rmse})
print(pd.DataFrame(rows_w).round(4).to_string(index=False))

 weighting  precision@10  recall@10  map@10  ndcg@10  hit_rate@10   rmse
      none        0.0427     0.0159  0.0179   0.0219       0.2813 1.1139
popularity        0.0336     0.0097  0.0143   0.0148       0.2240 1.1450


**Findings:**
Like the user side, the fix is principled and does not pay off here: every ranking metric is worse with popularity weighting. Once the model already requires 100 shared users per pair, the most spurious matches are gone, and silencing prolific users also discards genuine signal. On MovieLens-1M with this split it is not a win.

## Item-based vs the baselines

Same split, same k=10, all four models against each other. The user-based default is k=5, min_co_ratings=5 from the last stage.

In [9]:
content = ContentBasedRecommender().fit(train, items=movies)
ubcf = UserBasedRecommender(k_neighbors=5, min_co_ratings=5).fit(train)

comparison = pd.DataFrame({
    "popularity": evaluate_ranking(popularity, train, test, k=10),
    "content-based": evaluate_ranking(content, train, test, k=10),
    "user-based": evaluate_ranking(ubcf, train, test, k=10),
    "item-based": evaluate_ranking(ibcf, train, test, k=10),
}).T
print(comparison[cols].round(4).to_string())

pred_cols = {
    "model": ["popularity", "content-based", "user-based", "item-based"],
    "rmse": [
        evaluate_predictions(popularity, test)["rmse"],
        evaluate_predictions(content, test)["rmse"],
        evaluate_predictions(ubcf, test)["rmse"],
        evaluate_predictions(ibcf, test)["rmse"],
    ],
    "mae": [
        evaluate_predictions(popularity, test)["mae"],
        evaluate_predictions(content, test)["mae"],
        evaluate_predictions(ubcf, test)["mae"],
        evaluate_predictions(ibcf, test)["mae"],
    ],
}
print()
print(pd.DataFrame(pred_cols).round(4).to_string(index=False))

               precision@10  recall@10  map@10  ndcg@10  hit_rate@10
popularity           0.1040     0.0379  0.0565   0.0530       0.4548
content-based        0.0248     0.0116  0.0103   0.0129       0.1680
user-based           0.0429     0.0143  0.0184   0.0205       0.2834
item-based           0.0427     0.0159  0.0179   0.0219       0.2813



        model   rmse    mae
   popularity 1.1464 0.9439
content-based 1.1464 0.9439
   user-based 1.1710 0.9515
   item-based 1.1139 0.8856


**Findings:**
Item-based and user-based land within noise of each other on ranking, both beating content-based and both trailing popularity. The item model's real edge is on rating prediction: RMSE 1.114 and MAE 0.886, clearly better than user-based (1.171, 0.951) and the only values under the popularity pair (1.146, 0.944). The honest caveat: that RMSE win comes mostly from the item-mean prior, since predicting each movie's mean alone scores 0.989. The similarity term is a small refinement on a strong prior, not the source of the gain.

## It does personalize

The aggregate hides the point of collaborative filtering: the lists are built from each user's own ratings. User 1's sister movies lead to war films and classic dramas; user 42's lead to comedy and animation. Their top-10s share nothing.

In [10]:
def show_titles(user_id, n=10):
    rec = ibcf.recommend(user_id, k=n)
    titled = pd.DataFrame({"movie_id": rec}).merge(
        movies[["movie_id", "title", "genres"]], on="movie_id"
    )
    print(f"user {user_id}")
    print(titled[["title", "genres"]].to_string(index=False))
    print()

show_titles(1)
show_titles(42)
overlap = set(ibcf.recommend(1, k=10)) & set(ibcf.recommend(42, k=10))
print(f"shared top-10 between users 1 and 42: {len(overlap)} of 10")

user 1
                               title                                   genres
Bridge on the River Kwai, The (1957)                             [Drama, War]
          Killing Fields, The (1984)                             [Drama, War]
           American History X (1998)                                  [Drama]
        Sense and Sensibility (1995)                         [Drama, Romance]
 Mr. Smith Goes to Washington (1939)                                  [Drama]
  Searching for Bobby Fischer (1993)                                  [Drama]
                     Fantasia (1940)         [Animation, Children's, Musical]
         Beauty and the Beast (1991)         [Animation, Children's, Musical]
                  October Sky (1999)                                  [Drama]
                      Aladdin (1992) [Animation, Children's, Comedy, Musical]

user 42
                                 title                                   genres
             Dead Poets Society (1989)        

**Findings:**
User 1 gets war, classic drama, and history; user 42 gets comedy, animation, and cult classics, and the lists share zero movies. The sister movies are doing the work. This is personalization without a single genre feature in the model, and it is the same personalization the popularity baseline can never produce.

## New-item cold start

The mirror image of user-based's new-user problem. A movie that nobody in the training split rated has no sisters to lean on, so the model has nothing to say about it. The catalog contains 216 movies with zero train ratings, and a brand-new movie behaves like them.

In [11]:
rated = set(train["movie_id"])
unrated = movies.loc[~movies["movie_id"].isin(rated), "movie_id"]
print(f"catalog movies with zero ratings in train: {len(unrated)}")

# a movie id that does not exist in the training split at all
print(f"predict for a brand-new movie:  {ibcf.predict(1, 999999):.4f}  (global mean)")
print(f"recommend for a brand-new user: {ibcf.recommend(999999, k=5)}")

catalog movies with zero ratings in train: 216
predict for a brand-new movie:  3.6169  (global mean)
recommend for a brand-new user: []


**Findings:**
A brand-new movie gets the global mean and cannot be ranked, and a brand-new user gets the empty list, exactly the failure modes of the user-based stage transposed. User-based needed the user's history, item-based needs the item's history, and both need a rating matrix to work from. The two collaborative filtering stages have now documented the same wall from both sides: taste without history, and history without taste, are both unreadable.